# infinaxis controls demo

This notebook shows the main `Axis` and `GcodeControls` options that affect generated gcode.

The next cell contains reusable functions that help keep subsequent cells concise

In [ ]:
if 'google.colab' in str(get_ipython()):
  !pip install git+https://github.com/FullControlXYZ/fullcontrol --quiet
import lab.fullcontrol.infinaxis as fci

EW = 0.6
EH = 0.3
print_settings = {'extrusion_width': EW, 'extrusion_height': EH}

base_steps_data = [
    dict(x=0, y=10, z=EH / 2, b=0, c=0),
    dict(x=4, y=10, z=EH / 2, b=10, c=20),
    dict(x=8, y=12, z=EH, b=20, c=40),
    dict(x=12, y=14, z=EH * 1.5, b=30, c=60),
]

def build_steps(head_chain, bed_chain, steps_data=base_steps_data):
    Point = fci.configure_point(head_chain, bed_chain)
    return [Point(**step_data) for step_data in steps_data]

def controls(head_chain, bed_chain, **kwargs):
    return fci.GcodeControls(
        head_chain=head_chain,
        bed_chain=bed_chain,
        initialization_data=print_settings,
        **kwargs,
    )

def print_gcode_sample(label, steps, gcode_controls, look_for):
    gcode = fci.transform(steps, 'gcode', gcode_controls)
    print('___\n' + label)
    print(f'Look for: {look_for}')
    print('\n'.join(gcode.split('\n')[-8:]))


## Baseline

A normal two-rotary-axis example. `Axis.name` is the gcode name and, for standard `A/B/C`, also provides the default kinematic `type`.

In [ ]:
head_chain = [fci.Axis(name='B')]
bed_chain = [fci.Axis(name='C')]
steps = build_steps(head_chain, bed_chain)

print_gcode_sample(
    'baseline Axis(name="B") + Axis(name="C"):',
    steps,
    controls(head_chain, bed_chain),
    'B and C appear in each move; XYZ is inverse-kinematics output.',
)


## Axis.name and Axis.type

Use `name` for the emitted gcode axis and `type` for the kinematic behavior. This allows machine names like `CI` and `CII` to both behave as C-style rotary axes.

In [ ]:
head_chain = [fci.Axis(name='CII', type='C')]
bed_chain = [fci.Axis(name='CI', type='C')]
Point = fci.configure_point(head_chain, bed_chain)
steps = [
    Point(x=0, y=10, z=EH / 2, CI=0, CII=0),
    Point(x=4, y=10, z=EH / 2, CI=20, CII=-10),
    Point(x=8, y=12, z=EH, CI=40, CII=-20),
]

print_gcode_sample(
    'custom axis names with shared C kinematics:',
    steps,
    controls(head_chain, bed_chain),
    'CI and CII are emitted, while type="C" controls the rotation math.',
)


## Axis.active

`active` sets the starting position of an axis before any point explicitly changes it.

### QUESTION: is 'active' necessary? It seems like you could do that by setting axis to have an initial value in the first Point. I assume this is actually to allow the axis to have a non-zero value when setting up subsequent axes after it in the same head or bed chain? If so, perhaps a better name than 'active' exists? Either way, give a clear description in code comments and in this demo notebook

In [ ]:
head_chain = []
bed_chain = [fci.Axis(name='C', active=45)]
Point = fci.configure_point(head_chain, bed_chain)
steps = [
    Point(x=0, y=10, z=EH / 2),
    Point(x=4, y=10, z=EH / 2),
    Point(x=8, y=12, z=EH, c=90),
]

print_gcode_sample(
    'Axis(active=45) before points set c:',
    steps,
    controls(head_chain, bed_chain),
    'C starts at 45.0, then changes to 90.0 when the point sets c=90.',
)


## Axis.orientation

`orientation` flips the sign used by the kinematic calculation. The commanded axis value is still emitted with the configured `Axis.name`; the XYZ solution changes.

In [ ]:
orientation_steps_data = [
    dict(x=0, y=10, z=EH / 2, c=0),
    dict(x=4, y=10, z=EH / 2, c=20),
    dict(x=8, y=12, z=EH, c=40),
]

for orientation in [1, -1]:
    head_chain = []
    bed_chain = [fci.Axis(name='C', orientation=orientation)]
    steps = build_steps(head_chain, bed_chain, orientation_steps_data)
    print_gcode_sample(
        f'Axis(name="C", orientation={orientation}):',
        steps,
        controls(head_chain, bed_chain),
        'Compare XYZ values between orientation=1 and orientation=-1. C values remain commanded values.',
    )


## Axis.offset

`offset` describes the location of an axis relative to the previous axis in the chain. CHECK_THIS_DESCRIPTION: confirm exact sign convention for the target machine before using this on hardware.

In [ ]:
for offset in [fci.Point(x=0, y=0, z=0), fci.Point(x=5, y=0, z=0)]:
    head_chain = [fci.Axis(name='B', offset=offset)]
    bed_chain = [fci.Axis(name='C')]
    steps = build_steps(head_chain, bed_chain)
    print_gcode_sample(
        f'Axis(name="B", offset={offset}):',
        steps,
        controls(head_chain, bed_chain),
        'Compare XYZ values as the head rotary axis offset changes.',
    )


## GcodeControls.verbose

`verbose=True` adds distance diagnostics as gcode comments.

In [ ]:
head_chain = [fci.Axis(name='B')]
bed_chain = [fci.Axis(name='C')]
steps = build_steps(head_chain, bed_chain)

for verbose in [False, True]:
    print_gcode_sample(
        f'GcodeControls(verbose={verbose}):',
        steps,
        controls(head_chain, bed_chain, verbose=verbose),
        'Verbose output adds ; distance and system comments when enabled.',
    )


## GcodeControls.distance_axis

`distance_axis=True` emits a synthetic `U` value based on accumulated model/path distance.

In [ ]:
head_chain = [fci.Axis(name='B')]
bed_chain = [fci.Axis(name='C')]
steps = build_steps(head_chain, bed_chain)

print_gcode_sample(
    'GcodeControls(distance_axis=True):',
    steps,
    controls(head_chain, bed_chain, distance_axis=True),
    'U appears after B/C and increases as accumulated distance changes.',
)


## GcodeControls.model_XYZ_gcode

`model_XYZ_gcode=True` emits model-space `U/V/W` values. CHECK_THIS_DESCRIPTION: confirm target firmware/motion-planner expectations.

In [ ]:
head_chain = [fci.Axis(name='B')]
bed_chain = [fci.Axis(name='C')]
steps = build_steps(head_chain, bed_chain)

print_gcode_sample(
    'GcodeControls(model_XYZ_gcode=True):',
    steps,
    controls(head_chain, bed_chain, model_XYZ_gcode=True),
    'U/V/W show the model-space x/y/z values alongside transformed XYZ/B/C.',
)


## GcodeControls.inverse_time_feedrate

`inverse_time_feedrate=True` changes the F calculation for extrusion moves. CHECK_THIS_DESCRIPTION: verify units and firmware mode before using on a machine.

### QUESTION: Don't we need to add an M command at the start of the gcode if this is set to true? That should be automatically added (like M83)

In [ ]:
head_chain = [fci.Axis(name='B')]
bed_chain = [fci.Axis(name='C')]
steps = build_steps(head_chain, bed_chain)

print_gcode_sample(
    'GcodeControls(inverse_time_feedrate=True):',
    steps,
    controls(head_chain, bed_chain, inverse_time_feedrate=True),
    'F values differ from the baseline calculation.',
)


## GcodeControls.xyz_orientation

`xyz_orientation` flips output signs for system XYZ axes. CHECK_THIS_DESCRIPTION: confirm this is the intended way to handle machine axis direction.

In [ ]:
head_chain = [fci.Axis(name='B')]
bed_chain = [fci.Axis(name='C')]
steps = build_steps(head_chain, bed_chain)

print_gcode_sample(
    'Baseline output for regular orientation, GcodeControls(xyz_orientation=[1, 1, 1]):',
    steps,
    controls(head_chain, bed_chain, xyz_orientation=[1, 1, 1]),
    '',
)

print_gcode_sample(
    'X orientation flipped, GcodeControls(xyz_orientation=[-1, 1, 1]):',
    steps,
    controls(head_chain, bed_chain, xyz_orientation=[-1, 1, 1]),
    'X output changes sign compared with the baseline.',
)
